In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# Store（长期记忆）
用于在不同会话间共享数据

## Store的数据结构
Store的数据格式是JSON文档，JSON文档采用分级管理：
- Namespace（命名空间）：理解为一个文件夹
    - Key（键）：理解为文件名，但必须唯一
    - Value（值）：要存储的JSON文档

In [2]:
from langgraph.store.memory import InMemoryStore

memory_store = InMemoryStore()

# 向store中存储数据
memory_store.put(
    ("preferences", ),   # namespace 是一个tuple
    "user_001",  # key 可以是任意类型
    {
        # value, 是JSON格式文档
        "style": "business_markdown",
        "language": "zh-CN"
    }
)

memory_store.put(
    ("preferences", ), 
    "user_002", 
    {
        "style": "trump",
        "language": "en-US"
    }
)

读取store数据：  
- get：在指定namespace下根据key查找
- search：在指定namespace下对value做语义搜索或者过滤

In [3]:
# 基于get查询
user_preferences = memory_store.get(
    # namespace
    ("preferences", ),
    # key
    "user_001"
)

# 获取key后再获取value
print(f"用户信息：{user_preferences.value if user_preferences else "Not found"}")

用户信息：{'style': 'business_markdown', 'language': 'zh-CN'}


In [4]:
# 基于search搜索数据
search_result = memory_store.search(
    ("preferences", ),
    filter={"language": "zh-CN"},
    limit=5
)

print(f"搜索结果数量：{len(search_result)}")
print(search_result)

搜索结果数量：1
[Item(namespace=['preferences'], key='user_001', value={'style': 'business_markdown', 'language': 'zh-CN'}, created_at='2026-07-31T02:07:59.836188+00:00', updated_at='2026-07-31T02:07:59.836188+00:00', score=None)]


## 基于向量模型的store

In [5]:
# 定义基于向量的store
from langgraph_cli.schemas import IndexConfig
from langchain_community.embeddings import DashScopeEmbeddings
import os

# 初始化向量模型
embedding_model = DashScopeEmbeddings(
    model = "qwen3.7-text-embedding",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 初始化store
memory_store = InMemoryStore(
    index=IndexConfig(
        embed=embedding_model,  # 向量模型
        dims=1024  # 向量维度
    )
)

In [6]:
memory_store.put(
    ("users",), 
    "user_001", 
    {
        "id": "user_001",
        "name": "张三",
        "department": "技术部",
        "clearance_level": 3
    }
)

memory_store.put(
    ("users",), 
    "user_002", 
    {
        "id": "user_002",
        "name": "李四",
        "department": "市场部",
        "clearance_level": 1
    }
)

In [8]:
# 读取store是可以选择基于向量相似度的搜索方式
# 基于get查询
user_data = memory_store.get(
    ("users", ),
    "user_001"
)
print(f"用户信息：{user_data.value if user_data else 'Not found'}")

用户信息：{'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}


In [10]:
# 基于search搜索数据
search_result = memory_store.search(
    ("users", ),
    # 基于字段语义检索
    query="001",  # 将"001"转化为向量，然后在命名空间内的所有向量中进行相似度搜索
    limit=5  # 找到最相似的5个记忆
)

print(f"搜索结果数量：{len(search_result)}")
print(search_result)

搜索结果数量：2
[Item(namespace=['users'], key='user_001', value={'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}, created_at='2026-07-31T02:28:24.220433+00:00', updated_at='2026-07-31T02:28:24.220433+00:00', score=0.357011092744243), Item(namespace=['users'], key='user_002', value={'id': 'user_002', 'name': '李四', 'department': '市场部', 'clearance_level': 1}, created_at='2026-07-31T02:28:24.601301+00:00', updated_at='2026-07-31T02:28:24.601301+00:00', score=0.35406779835853985)]


## 在tool中访问store
与state类似，在tool 中访问store也是通过runtime

In [11]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """获取用户信息"""
    if runtime.store is None:
        return "Store is not available"

    # 通过runtime获取store，读取其中的数据
    user_info = runtime.store.get(
        ("users", ),
        user_id
    )

    if user_id is None:
        return "没有找到用户"

    return f"用户信息：{user_info.value}"

In [12]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-chat",
    tools=[get_user_info],
    store=memory_store  # 指定store的存储方式
)

response = agent.invoke(
    {"messages": [HumanMessage("帮我查询user_001的信息")]}
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查询user_001的信息
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_yA0u4RpY3l8KtexOLfZ90055)
 Call ID: call_00_yA0u4RpY3l8KtexOLfZ90055
  Args:
    user_id: user_001
================================= Tool Message =================================
Name: get_user_info

用户信息：{'id': 'user_001', 'name': '张三', 'department': '技术部', 'clearance_level': 3}
================================== Ai Message ==================================

已经为您查询到用户 **user_001** 的信息，具体如下：

| 字段 | 信息 |
|------|------|
| **用户ID** | user_001 |
| **姓名** | 张三 |
| **部门** | 技术部 |
| **权限等级** | 3 |

如果您还需要查询其他用户的信息，请随时告诉我。
